In [7]:
import sys; sys.path.append("../src")
import pandas as pd, numpy as np
from pathlib import Path
from metrics import evaluate

files = {f.name: f for f in Path("../data").rglob("*.csv")}
print(files.keys())

tx = pd.read_csv(files["txs_features.csv"])
print([c for c in tx.columns if "BTC" in c or "fee" in c.lower() or "ime" in c])
rng = np.random.default_rng(42)
results.append(evaluate("Random (reference)", y, rng.random(len(y)), 0.99))
pd.DataFrame(results)

dict_keys(['elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv', 'elliptic_txs_features.csv', 'txs_classes.csv', 'txs_edgelist.csv', 'txs_features.csv'])
['Time step', 'total_BTC', 'fees', 'in_BTC_min', 'in_BTC_max', 'in_BTC_mean', 'in_BTC_median', 'in_BTC_total', 'out_BTC_min', 'out_BTC_max', 'out_BTC_mean', 'out_BTC_median', 'out_BTC_total']


,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Amount threshold (baseline),0.00,0.00,0.00,0.044,1.19
1,Random (reference),0.02,0.05,0.07,0.066,1.14


In [8]:
TIME, AMOUNT = "Time step", "total_BTC"

cls = pd.read_csv(files["txs_classes.csv"])
tx = tx.merge(cls, on="txId")
tx["y"] = tx["class"].astype(str).map({"1": 1, "2": 0})   # anything else = unknown

train = tx[tx[TIME] <= 34]
test  = tx[tx[TIME] >= 35]
test_lab = test[test["y"].notna()]

print("Train:", len(train), "| Test:", len(test), "| Labeled test:", len(test_lab))
print("Illicit share in labeled test:", round(test_lab["y"].mean(), 3))

Train: 136265 | Test: 67504 | Labeled test: 16670
Illicit share in labeled test: 0.065


In [9]:
threshold = train[AMOUNT].quantile(0.99)   # top 1% of training amounts = "suspicious"

y      = test_lab["y"].to_numpy()
scores = test_lab[AMOUNT].fillna(0).to_numpy()

results = [evaluate("Amount threshold (baseline)", y, scores, threshold)]
pd.DataFrame(results)

,model,P@50,P@100,P@500,PR-AUC,flag_rate_%
0,Amount threshold (baseline),0.0,0.0,0.0,0.044,1.19


In [10]:
print("NaN in y:", np.isnan(y).sum())
print("NaN in scores:", np.isnan(scores).sum())
print(test_lab[AMOUNT].dtype)

NaN in y: 0
NaN in scores: 0
float64
